# AIMNet2: exposing the electrostatic coupling head

Load an AIMNet2 calculator, inspect its PyTorch model, and add the Coulomb coupling term as a separate output. This term is useful when studying charge response: its derivative with respect to atomic charges is an electrostatic potential contribution. This notebook exposes the energy term; it does not calculate a polarizability.

Use the demo's `aimnet2` kernel. The first run may download model weights. Run cells in order from a fresh kernel.

In [1]:
import os
import sys

# On macOS, avoid mixed-architecture TorchInductor caches. Set before importing torch.
if sys.platform == 'darwin':
    os.environ.setdefault('TORCHDYNAMO_DISABLE', '1')

import copy
import torch
from torch import nn
from ase.build import molecule
from aimnet.calculators import AIMNet2Calculator

calculator = AIMNet2Calculator('aimnet2', device='cpu')
model = calculator.model
print('Calculator model type:', type(model))
print(model)

Calculator model type: <class 'aimnet.models.aimnet2.AIMNet2'>
AIMNet2(
  (aev): AEVSV()
  (afv): Embedding(64, 16, padding_idx=0)
  (conv_a): ConvSV()
  (conv_q): ConvSV()
  (mlps): ModuleList(
    (0): Sequential(
      (0): Linear(in_features=704, out_features=512, bias=True)
      (1): GELU(approximate='none')
      (2): Linear(in_features=512, out_features=380, bias=True)
      (3): GELU(approximate='none')
      (4): Linear(in_features=380, out_features=258, bias=True)
    )
    (1): Sequential(
      (0): Linear(in_features=733, out_features=512, bias=True)
      (1): GELU(approximate='none')
      (2): Linear(in_features=512, out_features=380, bias=True)
      (3): GELU(approximate='none')
      (4): Linear(in_features=380, out_features=258, bias=True)
      (5): GELU(approximate='none')
    )
    (2): Sequential(
      (0): Linear(in_features=733, out_features=512, bias=True)
      (1): GELU(approximate='none')
      (2): Linear(in_features=512, out_features=380, bias=True)
  

## Find the coupling module

Some AIMNet2 models put `lrcoulomb` in `model.outputs`. Current models may attach the long range Coulomb module to the calculator and put a short range correction in `model.outputs`. Select the module actually used by the loaded calculator, so its parameters and cutoff match that model.

In [1]:
from McUtils.Jupyter import JHTML

In [4]:
JHTML.JAlert(
    "Not every AIMNet2 model implements charges in the same way!",
    kind="error"
)

In [16]:
JHTML.JCard(
    "Log in to ",
    JHTML.Code("JupyterLab"),
    " choose ",
    JHTML.Code("launch"),
    ". After a brief startup you'll see a window that looks like",
    JHTML.Pre("""
This is sample text, not real
Lorem ipsum, etc.
Close the window after you're done so we don't get ants
"""), 
    title='Running a Calculation', 
    kind='primary'
).to_copy_button()

<div style="border-radius:2px; overflow:hidden; border:1px solid #dee2e6;">
  <div style="font-weight:600; padding:10px 14px; border-bottom:1px solid #b8daff; background-color:#cce5ff; color:#004085;">Running a Calculation</div>
  <div style="padding:14px;">Log in to <code>JupyterLab</code> choose <code>launch</code>. After a brief startup you'll see a window that looks like<pre>
This is sample text, not real
Lorem ipsum, etc.
Close the window after you're done so we don't get ants
</pre></div>
</div>

In [2]:
if not isinstance(model.outputs, nn.ModuleDict):
    raise TypeError('This example expects model.outputs to be a torch.nn.ModuleDict')

print('Model output modules:', list(model.outputs))
embedded = model.outputs['lrcoulomb'] if 'lrcoulomb' in model.outputs else None
external = getattr(calculator, 'external_coulomb', None)

if embedded is not None:
    coupling_head = embedded
    source = "model.outputs['lrcoulomb']"
elif external is not None:
    coupling_head = external
    source = 'calculator.external_coulomb'
else:
    raise RuntimeError('No Coulomb coupling module found on this AIMNet2 calculator')

print('Coupling module source:', source)
print('Coupling module:', coupling_head)
print('Input key:', coupling_head.key_in, 'original output key:', coupling_head.key_out)
if 'srcoulomb' in model.outputs:
    print('Short range correction:', model.outputs['srcoulomb'])

Model output modules: ['energy_mlp', 'atomic_shift', 'atomic_sum', 'srcoulomb']
Coupling module source: calculator.external_coulomb
Coupling module: LRCoulomb()
Input key: charges original output key: energy
Short range correction: SRCoulomb()


## Baseline calculation

Use the calculator's dictionary interface to retrieve model outputs. The ASE wrapper exposes a fixed set of properties and will not return this additional key.

In [3]:
water = molecule('H2O')
inputs = {'coord': water.positions, 'numbers': water.numbers, 'charge': 0.0}
baseline = calculator.eval(inputs)
print('Baseline energy (eV):', baseline['energy'].detach().cpu().numpy())
print('Atomic charges (e):', baseline['charges'].detach().cpu().numpy())

Baseline energy (eV): [-2081.04087582]
Atomic charges (e): [-0.6898448  0.3449224  0.3449224]


## Register a separate output head

Copy the loaded coupling module and change the copy's output key. The original energy contribution remains in place. AIMNet2 calls output modules in insertion order, so the new head runs after the existing heads. The calculator filters returned keys, so add the new key to its output list too.

In [4]:
output_key = 'electrostatic_energy'
head_name = 'electrostatic_probe'
if head_name in model.outputs or output_key in calculator.keys_out:
    raise RuntimeError('Probe already registered; restart the kernel before rerunning this cell')

probe = copy.deepcopy(coupling_head)
probe.key_out = output_key
model.outputs[head_name] = probe
calculator.keys_out = [*calculator.keys_out, output_key]

print('Registered:', head_name, '->', model.outputs[head_name])
print('Returned keys:', calculator.keys_out)

Registered: electrostatic_probe -> LRCoulomb()
Returned keys: ['energy', 'charges', 'spin_charges', 'forces', 'hessian', 'stress', 'electrostatic_energy']


In [5]:
result = calculator.eval(inputs)
print('Returned keys:', list(result))
print('Total energy (eV):', result['energy'].detach().cpu().numpy())
print('Electrostatic coupling energy (eV):', result[output_key].detach().cpu().numpy())

torch.testing.assert_close(result['energy'], baseline['energy'])
torch.testing.assert_close(result['charges'], baseline['charges'])
assert result[output_key].numel() == result['energy'].numel()
print('Verified: total energy and atomic charges are unchanged.')

Returned keys: ['charges', 'energy', 'electrostatic_energy']
Total energy (eV): [-2081.04087582]
Electrostatic coupling energy (eV): [-5.95270205]
Verified: total energy and atomic charges are unchanged.


The separate value is the selected Coulomb module's energy contribution, in eV per structure. With the current default model, this is the external full Coulomb term, while `srcoulomb` supplies a short range subtraction in the total energy. With an embedded `lrcoulomb` model, it is that embedded head's term. Access the added output on later calls with `calculator.eval(inputs)['electrostatic_energy']`.